In [1]:
import os

os.environ["HF_HOME"] = "C:/hf_cache"
os.environ["HF_DATASETS_CACHE"] = "C:/hf_cache/datasets"
os.environ["HF_HUB_CACHE"] = "C:/hf_cache/hub"

from datasets import load_dataset

dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")
print(dataset)

C:\Users\Bishwa Bolt\PycharmProjects\customer-support-llm-finetuning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['flags', 'instruction', 'category', 'intent', 'response'],
        num_rows: 26872
    })
})


In [2]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_qwen_prompt(example):
    messages = [
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["response"]},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": prompt}

formatted_dataset = dataset["train"].map(format_qwen_prompt)
print(formatted_dataset[0]["text"])

split_dataset = formatted_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print("\nTrain examples:", len(train_dataset))
print("Eval examples:", len(eval_dataset))

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
question about cancelling order {{Order Number}}<|im_end|>
<|im_start|>assistant
I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you.<|im_end|>


Train examples: 24184
Eval examples: 2688


In [3]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print("Model loaded successfully")
print("Memory footprint (GB):", round(model.get_memory_footprint() / (1024**3), 2))

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

C:\Users\Bishwa Bolt\PycharmProjects\customer-support-llm-finetuning\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\hf_cache\hub\models--Qwen--Qwen2.5-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Model loaded successfully
Memory footprint (GB): 1.48
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [4]:
MAX_LENGTH = 512

def tokenize_function(example):
    tokenized = tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_train = train_dataset.map(
    tokenize_function,
    remove_columns=train_dataset.column_names,
    batched=False,
)
tokenized_eval = eval_dataset.map(
    tokenize_function,
    remove_columns=eval_dataset.column_names,
    batched=False,
)

print("Tokenized train examples:", len(tokenized_train))
print("Tokenized eval examples:", len(tokenized_eval))
print("Sample input_ids length:", len(tokenized_train[0]["input_ids"]))

Map: 100%|██████████| 2688/2688 [00:02<00:00, 1263.12 examples/s]

Tokenized train examples: 24184
Tokenized eval examples: 2688
Sample input_ids length: 512


In [5]:
import mlflow

mlflow.set_tracking_uri("file:///" + os.path.abspath("../mlflow/mlruns").replace("\\", "/"))
mlflow.set_experiment("qwen2.5-1.5b-qlora-customer-support")

print("MLflow tracking URI:", mlflow.get_tracking_uri())

2026/07/13 09:54:47 INFO mlflow.tracking.fluent: Experiment with name 'qwen2.5-1.5b-qlora-customer-support' does not exist. Creating a new experiment.


MLflow tracking URI: file:///C:/Users/Bishwa Bolt/PycharmProjects/customer-support-llm-finetuning/mlflow/mlruns


In [6]:
tokenized_train_small = tokenized_train.shuffle(seed=42).select(range(6000))
tokenized_eval_small = tokenized_eval.shuffle(seed=42).select(range(600))

print("Reduced train examples:", len(tokenized_train_small))
print("Reduced eval examples:", len(tokenized_eval_small))

Reduced train examples: 6000
Reduced eval examples: 600


In [7]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="../qwen_qlora_checkpoints",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,
    gradient_checkpointing=False,
    fp16=True,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=300,
    save_strategy="steps",
    save_steps=300,
    save_total_limit=2,
    warmup_steps=30,
    optim="adamw_bnb_8bit",
    report_to=[],
    max_grad_norm=0.3,
    logging_dir="../logs",
    dataloader_num_workers=2,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_small,
    eval_dataset=tokenized_eval_small,
    data_collator=data_collator,
)

print("Optimized trainer initialized")
print("Total training steps:", len(tokenized_train_small) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps) * training_args.num_train_epochs)

Optimized trainer initialized
Total training steps: 1500


C:\Users\Bishwa Bolt\PycharmProjects\customer-support-llm-finetuning\.venv\Lib\site-packages\accelerate\accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [8]:
model.enable_input_require_grads()

trainer.args.max_steps = 20

with mlflow.start_run(run_name="qwen_smoke_test_20_steps"):
    train_result = trainer.train()
    print("Smoke test completed successfully")
    print("Final train loss:", train_result.training_loss)

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
C:\Users\Bishwa Bolt\PycharmProjects\customer-support-llm-finetuning\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss


Smoke test completed successfully
Final train loss: 1.9404905319213868


In [9]:
import torch, gc
try:
    del trainer
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared")

GPU memory cleared


In [10]:
tokenized_train_small = tokenized_train.shuffle(seed=42).select(range(12000))
tokenized_eval_small = tokenized_eval.shuffle(seed=42).select(range(1200))

print("Reduced train examples:", len(tokenized_train_small))
print("Reduced eval examples:", len(tokenized_eval_small))

Reduced train examples: 12000
Reduced eval examples: 1200


In [11]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="../qwen_qlora_checkpoints",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,
    gradient_checkpointing=False,
    fp16=True,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=300,
    save_strategy="steps",
    save_steps=300,
    save_total_limit=2,
    warmup_steps=30,
    optim="adamw_bnb_8bit",
    report_to=[],
    max_grad_norm=0.3,
    logging_dir="../logs",
    dataloader_num_workers=2,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_small,
    eval_dataset=tokenized_eval_small,
    data_collator=data_collator,
)

model.enable_input_require_grads()

print("Trainer rebuilt for 12K run")
print("Total training steps:", len(tokenized_train_small) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps) * training_args.num_train_epochs)

Trainer rebuilt for 12K run
Total training steps: 3000


C:\Users\Bishwa Bolt\PycharmProjects\customer-support-llm-finetuning\.venv\Lib\site-packages\accelerate\accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [12]:
with mlflow.start_run(run_name="qwen_qlora_12k_run"):
    mlflow.log_params({
        "model_name": "Qwen/Qwen2.5-1.5B-Instruct",
        "lora_r": 16,
        "lora_alpha": 32,
        "learning_rate": 2e-4,
        "batch_size_per_device": 2,
        "gradient_accumulation_steps": 2,
        "effective_batch_size": 4,
        "max_seq_length": 512,
        "num_epochs": 1,
        "train_examples": len(tokenized_train_small),
    })

    train_result = trainer.train()

    mlflow.log_metric("final_train_loss", train_result.training_loss)

    print("Training completed successfully")
    print("Final train loss:", train_result.training_loss)

C:\Users\Bishwa Bolt\PycharmProjects\customer-support-llm-finetuning\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
300,0.719700,0.733025
600,0.675700,0.668549
900,0.606800,0.641591
1200,0.641300,0.622341
1500,0.633100,0.604981
1800,0.595100,0.591107
2100,0.619800,0.580497
2400,0.607300,0.571974
2700,0.548300,0.564004
3000,0.554300,0.560180


We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)
C:\Users\Bishwa Bolt\PycharmProjects\customer-support-llm-finetuning\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
C:\Users\Bishwa Bolt\PycharmProjects\customer-support-llm-finetuning\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. 

Training completed successfully
Final train loss: 0.6319411075909932


In [13]:
ADAPTER_SAVE_PATH = "../qwen_qlora_adapter"

model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)

print("Adapter and tokenizer saved to:", ADAPTER_SAVE_PATH)

import os
print("\nFiles saved:")
for f in os.listdir(ADAPTER_SAVE_PATH):
    print(" -", f)

Adapter and tokenizer saved to: ../qwen_qlora_adapter

Files saved:
 - adapter_config.json
 - adapter_model.safetensors
 - added_tokens.json
 - merges.txt
 - README.md
 - special_tokens_map.json
 - tokenizer.json
 - tokenizer_config.json
 - vocab.json
